In [19]:
##### IMPORTS

import os
os.environ['picaso_refdata'] = r'C:\Users\Alex\Desktop\Picaso\picaso\reference' # THIS MUST GO BEFORE YOUR IMPORT STATEMENT
os.environ['PYSYN_CDBS'] = r'C:\Users\Alex\Desktop\Picaso\grp\redcat\trds' # This is for the stellar data discussed below.

# General
import numpy as np
import astropy.units as u
import bd_support as sup
from pathlib import Path
from itertools import product

# Picaso and Virga
from picaso import justdoit as jdi
from virga import justdoit as vj

# To see what clouds are availible
# vj.available()

In [20]:
##### CONFIGURATIONS

# Directories
sonor_path  = r'C:\Users\Alex\Desktop\Picaso\data\sonora' # Sonora db
# sonor_path  = '/groups/tkaralidi/pbraunschweig/training_set/profiles/'
virga_path  = r'C:\Users\Alex\Desktop\Picaso\data\virga'  # Virga
# virga_path  = '/home/sa221179/picaso/virga/'
opaci_path  = None # Opacity db
# opcai_path  = '/groups/tkaralidi/opacity_500k_for_R5000_egpoutput.db'
output_path = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs")
# output_path = 'home/al864695/ouputs'
pickl_path  = r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\bobcat_to_diamondback.pickle"
# pickl_path  = "home/al864695/pickle"

# Things that will remain constant
clouds      = ['Al2O3', 'CaTiO3', 'CH4', 'Cr', 'Fe',
               'H2O', 'KCl', 'Mg2SiO4', 'MgSiO3', 'MnS',
               'NH3', 'Na2S', 'TiO2', 'ZnS']

# Constant values
wav_range   = [0.3, 5.0] # microns
MH          = 1.0        # [M/H] metallicity factor ~ solar
MU          = 2.36       # Average MU
R           = 300        # resolution
# R           = 5000

# TP profile setup
# corr[301, 41, 5, 91]
Teff_grid = np.linspace(500, 2000, 301)          # length 301, in K
logg_grid = np.linspace(3.5, 5.55, 41)           # length 41, in m s^-2
fsed_grid = [1, 2, 3, 4, 8]                      # length 5
kzz_vals  = np.array([1e6, 1e7, 1e8, 1e9, 1e10]) # cm^2 s^-1
corr      = sup.TPCorrection(pickl_path)
corr.set_coords(Teff_grid, logg_grid, fsed_grid)

In [21]:
##### METHOD BROWN DWARF SPECTRUM

def bd_spectrum(Teff, gravity, fsed, kzz, corr=None):
    """
    Compute a BD emission spectrum with Virga clouds.
    """

    # Opacity & inputs
    opa    = jdi.opannection(wav_range, opaci_path)
    bd     = jdi.inputs(calculation="browndwarf")
    bd.phase_angle(0)
    bd.gravity(gravity, gravity_unit=u.Unit('m/s**2'))
    bd.sonora(sonor_path, Teff)

    # Inject corrected TP
    _Tcorr = corr.apply_to_picaso_inputs(bd, Teff, gravity, fsed)

    # Inject Kzz (match pressure grid length)
    prof   = bd.inputs['atmosphere']['profile']
    P      = np.asarray(prof["pressure"], float)
    bd.inputs["atmosphere"]["profile"]["kz"] = [float(kzz)] * len(P)

    # Clouds
    bd.virga(clouds, virga_path, fsed, mh=MH, mmw=MU)
    out    = bd.spectrum(opa, full_output=True)

    # Convert to F_nu, then regrid to constant R in wavenumber
    wn, fl = out["wavenumber"], out["thermal"]  # cm^-1 and erg cm^-2 s^-1 cm^-1

    # Convert wavenumber [cm^-1] to wavelength [micron]
    w_um = 1e4 / wn

    # Sort to ascending wavelength
    idx    = np.argsort(w_um)
    w_um, flux = w_um[idx], fl[idx]

    # Save to output dictionary
    out['regridx']  = w_um
    out['regridy']  = flux

    return w_um, flux

In [ ]:
##### GENERATE AND SAVE SPECTRUM (random)

# Define ranges to match your grid
Teff_min, Teff_max = 1300, 1800
logg_min, logg_max = 3.0, 5.0
fsed_min, fsed_max = 2.0, 4.0
kzz_min,  kzz_max  = 1e9, 1e10

N_samples = 50  # how many random points to draw

Teff_r = np.random.uniform(Teff_min, Teff_max, N_samples)
logg_r = np.random.uniform(logg_min, logg_max, N_samples)
fsed_r = np.random.uniform(fsed_min, fsed_max, N_samples)
kzz_r  = np.random.uniform(kzz_min , kzz_max , N_samples)

for Teff_i, logg_i, fsed_i, kzz_i in zip(Teff_r, logg_r, fsed_r, kzz_r):
    # Convert log g [cgs] → grav [m s^-2]
    grav_i = 10**logg_i * 1e-2
    # Run spectrum
    W_i, F_i = bd_spectrum(Teff_i, grav_i, fsed_i, kzz_i, corr)

    fname = f"rand_t{int(Teff_i)}g{int(logg_i)}f{int(fsed_i)}{sup.format_kzz(kzz_i)}.npz"
    fpath = output_path / fname

    # Save
    np.savez_compressed(fpath,
                        x=np.array([Teff_i, logg_i, fsed_i, kzz_i], dtype=float),
                        y=F_i.astype(float),
                        wavelength_um=W_i.astype(float))

print(f"Saved {N_samples} random files to {output_path}")